# Glue and what can make queries slow

Glue/Spark demo part A showed how Spark reads files: folders, byte ranges, footers, row groups and
reading tasks. 

In Part B we will cover slow queries and some ways to fix them, first part during the tutorial, then as exercises.

In steps 1 to 3, write down your answer to **Predict** before you run anything (as a comment in the query cell, e.g.
`# my guess: 8 tasks, 1 reading`). Then run the query cell and the `sm` cell under it (the checklist and the tables), and
find what **How to read it** names. Remember, sm is not a spark/glue tool, it is our library to quickly extract internal glue data about query performance and summarize it.

**Behind? Steps 1, 2 and 3 don't depend on each other.** Run the setup (if your session stopped) and the cell under
"The data for section 1" (it defines `trips`), then go to any step. Inside a step, run the cells in order.

End with `%stop_session` (the last cell) to stop Glue billing you!

AWS Glue setup config. We specify 3 workers (Spark driver and 2 Spark executors) and stop the cluster after a 15-minute idle timeout to limit charges if you forget to stop session.

But: **DON'T FORGET TO STOP THE SESSION**

In [ ]:
%glue_version 5.1
%worker_type G.1X
%number_of_workers 3
%idle_timeout 15

We also ask Glue to load our sparkmeter script from S3, as Glue doesn't have bootstrap scripts (in contrast to EMR).

In [ ]:
%%configure
{"--extra-py-files": "s3://436c-2026w1/l7/student/sparkmeter.py"}

**Setup.** Run the cell below: it starts the Glue session (30 to 60 s the first time) and loads `sparkmeter`. It prints the Spark version and the number of slots (a slot is one executor core: it runs one task at a time).

**Wait for "Session (session UUID) has been created" message before executing the next steps!**

In [ ]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from pyspark.sql import functions as F
sc = SparkContext.getOrCreate()
spark = GlueContext(sc).spark_session
import sparkmeter as sm
sm.setup(spark)      # to keep your runs after the session: sm.setup(spark, save_to="s3://436c-glue-<your account>/sparkmeter/")

## 1. Three slow queries and their fixes

**The data for section 1: the June 2012 trips, in copies that differ in one thing**

In part A we looked at 36 month data, here we will use only 15,096,468 trips made in June 2012. 

As we already discussed better ways to organize and read data in Part A, Part B uses four different ways to organize the same data to help us to study different aspects of queries:

| Copy | How it is stored | Used in |
|---|---|---|
| `june-eight-rowgroups` | 1 file, 8 row groups: 7 of about 2 million trips and 1 of 1.1 million | steps 1 to 3, exercises |
| `june-one-rowgroup` | 1 file, 1 row group | step 1 |
| `june-by-type` | a folder partition per payment type, 1 file in each | exercise B1 |
| `june-salted` | a folder partition per payment type, several files in each | exercise B1 |

Every copy has the same 10 columns, the same compression, and pickup times marked
as UTC (part A, step 1).

Steps 1 to 3 use the first two copies; the other two are for exercise B1.

A folder partition is a folder such as `payment_type=2/` (part A called it a Parquet partition). `trips` in the code
reads `june-eight-rowgroups` by default, unless mentioned. 
The session has 8 slots: 2 executors with 4 executor
cores each, as many executors and slots as on L6's 2 EMR core nodes.

Steps (correspond to L6 checks):
- Step 1 is **divide** (are there enough reading tasks for the
slots?). 
- Step 2 is **wait** (does one task hold most of the rows?), and already shows **move**. 
- Step 3 is **move** and **wait** together (how many rows go into an `Exchange`, and does one task hold most of them?).

**Run the next cell: it also defines `trips`, used in steps 1 to 3. Check:** how many row groups does each file have?
(`sm.tree` prints the average rows per row group.)

In [ ]:
problems = "s3://436c-2026w1/l7/problems/"
trips = spark.read.parquet(problems + "june-eight-rowgroups/")      # the copy steps 1 to 3 read, unless a cell names another

**What exactly Spark does here?**

In [ ]:
# Answer: try first, then read.
# Not much yet: spark.read.parquet records what to read. Spark lists the files in the folder and reads the
# schema from one file's footer (a tiny one-task Spark job: that is the progress bar you may see, and what sm
# calls the schema job), but no trips are read. The trips are read only when an action (collect(), count(),
# show()) runs the query. sm.tree, next, reads only the folder listing and the footers.

Next cell uses our sm helper to preview folder partitioning and rowgroups structure:

In [ ]:
sm.tree(problems + "june-one-rowgroup/", row_groups=True)
print()
sm.tree(problems + "june-eight-rowgroups/", row_groups=True)

**Step 1. Input can't be split finely enough (L6 P1)**

**What the query does.** L6's query: the number of rides and the average total for each payment type. It reads the
copy with **one** row group. Part A, step 4 read a June file with one row group too, but that file had 17 columns and
local times. This copy differs from the next one only in its row groups.

**The principle** (L6). Each reading task needs its own readable part of the input: a row group, or a range of a
splittable file. When the input has fewer readable parts than slots, some slots sit idle, and the stage lasts as long
as it takes to read the biggest part. Adding nodes doesn't make that part faster to read.

**Predict:** the session has 8 slots. How many scan tasks, and how many reading tasks?

In [ ]:
rides_one_rowgroup = (spark.read.parquet(problems + "june-one-rowgroup/")   # the copy with one row group
                      .groupBy("payment_type")
                      .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_one_rowgroup.collect() #what happens on collect?

In [ ]:
# Answer: try first, then read.
# collect() is an action: Spark turns the plan into tasks, the executors read the file and add up the trips per
# payment type, and the 5 result rows come back to the driver (the Glue worker that runs the notebook's Spark
# code) and from there to the notebook, as a list of Row objects. That list is the output under the cell.
# collect() brings every row of the result to the driver, so use it only on small results.

The cell above is spark's own output which you would see on any platform. Usually we would need to look at the logs, or Spark UI (difficult to reach on Glue) to see the performance, but we will use our helper library to do that: 

In [ ]:
sm.capture("rides, one row group", rides_one_rowgroup)
sm.checklist("rides, one row group")
sm.stage_table("rides, one row group")

**How to read it.**

- **Result:** 5 rows, one per payment type. Cash (type 2) has 7,799,619 trips and card (type 1) 7,241,859: nearly all.
- **Checklist:** `1 file -> 8 scan tasks`, and `1 of 8 scan tasks still hold a row group`.
- **Stage table, `scan stage` row:** `tasks` 8, `tasks with rows` 1. The one reading task's `largest task rows` is
  all 15,096,468 trips. `longest task s` is that task, and the stage lasts as long as it does. The other 7 scan tasks
  read 0 rows: their slots have nothing else to do.

**Why.** Spark cuts the file into 8 byte ranges, about one per slot (part A, "How `spark.read` turns files into
tasks"). A range reads rows only if it holds the middle of a row group. This file has one row group, so one task reads
it all. More slots would add empty scan tasks, not reading tasks, so "throwing more cores at it" doesn't work.

**The fix: more row groups or files.** The copy below holds the same trips in 8 row groups. Same query, same slots.

In [ ]:
rides_eight_rowgroups = (trips                           # the same trips in 8 row groups; nothing else changes
                         .groupBy("payment_type")
                         .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_eight_rowgroups.collect()

In [ ]:
sm.capture("rides, eight row groups", rides_eight_rowgroups)
sm.checklist("rides, eight row groups")
sm.stage_table("rides, eight row groups")

**What changed, and why.**

- `8 of 8 scan tasks still hold a row group`: each byte range now holds the middle of one row group.
- `scan stage` row: 8 tasks with rows, about 2 million rows each. `longest task s` is close to `median task s`.
- The query is faster: compare the time in the first line of the two tables. The same result, sooner.

**How many reading tasks we would have with 16 slots (twice the executors)?**


**Hot key is not always a problem (L6 P2).** Cash is a hot key: 7.8 million of the 15.1 million trips. But for a count or
an average, each task adds up its own trips first and sends one row per payment type, so no huge amount of rows goes through Exchange. Look at `rows to Exchange` in the `scan stage` row: 40, that is 5 payment types from each of the 8 scan tasks.

In [ ]:
# Answer: try first, then read.
# With 16 slots (4 executors instead of 2), Spark would cut the file into about 16 byte ranges: a range is about
# (file size + 4 MiB) / slots, kept between 4 MiB and 128 MiB, so (231 MB + 4 MiB) / 16 is about 15 MB, and the
# 231 MB file gives about 16 ranges, so about 16 scan tasks. Still 8 reading tasks: the file has 8 row groups,
# about 2 million rows each, and their middles land in different ranges; only a range that holds the middle of a
# row group reads rows. The other scan tasks read 0 rows. More reading tasks need more row groups or files.

**Step 2. Joins on a hot key (L6 P2.2)**

**What the query does.** The number of rides and the average total for each payment type **name**. The names are in
a 5-row table (1 card, 2 cash, 3 no charge, 4 dispute, 5 unknown), so the query joins the trips to it, then groups by
name. The first cell turns broadcast off: `spark.sql.autoBroadcastJoinThreshold` set to `-1` tells Spark never to copy
a table to every executor by itself. This is just for a demo, usually you would want to keep this on and let Spark deal with it.

**The principle** (L6). A *join* matches the rows of two tables on a key. When both tables go into an `Exchange`, all
rows with the same key are normally assigned to the same task in the next stage.

**Predict:** how many trips go into the `Exchange`? After it, how many tasks get trips? Can all 8 slots get some?

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")      # never broadcast by itself (for this demo only)

names = (spark.read.schema("payment_type long, payment_name string")
         .option("header", True).csv("s3://436c-2026w1/l7/layouts/payment_names.csv"))
rides_by_name = (trips
                 .join(names, "payment_type")
                 .groupBy("payment_name")
                 .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_by_name.collect()

In [ ]:
sm.capture("rides by name, broadcast off", rides_by_name)
sm.stage_table("rides by name, broadcast off")

**How to read it.** Read the stage table from the top: one row per stage, by stage number.

- `scan stage`: 8 tasks read the trips. `rows to Exchange` is 15,096,468: every trip goes into the `Exchange`, by
  payment type, about 87 `MB to Exchange`.
- `another input`: 1 task reads 5-row payment names table and sends it into the `Exchange` too.
- `stage after an Exchange`: JOIN. `tasks with rows` is 3. `largest task rows` is about 7.8 million: the trips
  paid by cash, with their name row. The task with the card trips comes next (`median task rows`, about 7.2 million).
- `final totals`: 1 task adds up the totals per name.

**Why.** The `Exchange` sends each trip to the task for its payment type. 5 payment types give at most 5 tasks with
rows, and nearly all trips are cash or card. 

This is our **wait** check: the longest task holds most of the rows and stage waits for it.
More slots alone can't split a key. Exercise B2 spreads it by hand with a salt; adaptive execution can split a hot
shuffle partition by itself (the 🔍 cell after exercise B2).

**The fix: broadcast the small table.** Spark copies the small table to every executor, so the trips go into no
`Exchange`: each scan task joins its own trips. Spark does it unasked when it knows the table is 10 MiB or less, as for our name table. 

When Spark cannot know the size or we want a larger table broadcasted (10 MiB is conservative), ask for the broadcast in the code: `F.broadcast(names)`. The request works even with the setting still at `-1`.

In [ ]:
rides_by_name_broadcast = (trips
                           .join(F.broadcast(names), "payment_type")   # copy the 5 names to every executor
                           .groupBy("payment_name")
                           .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_by_name_broadcast.collect()

In [ ]:
sm.capture("rides by name, broadcast", rides_by_name_broadcast)
sm.stage_table("rides by name, broadcast")

**What changed, and why.**

- The result is the same: the same 5 names, the same counts.
- No `stage after an Exchange` for the join: the trips went into no `Exchange`. Instead, join ran inside the 8 scan tasks.
- The `scan stage` sends 40 rows into the `Exchange`, not 15,096,468: the group-by's partial totals with 5 names from
  each of the 8 scan tasks (as in step 1, it is still a hot key, but it doesn't cost us much anymore).
- The query is faster: compare the first lines of `sm` output

Put the setting back:

In [ ]:
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

**Step 3. A window query (L6 P3.1)**

**What the query does.** It finds the expensive trips: those that cost more than twice the average of their payment
type. A *window query* computes a value over a group of rows and writes it back on every row of that group. Here, each
trip gets the average total of its payment type next to it (`type_avg`), and the filter keeps the trips above twice
that. Broadcast is back at its default.

**The principle** (L6 P3). Every row that goes into an `Exchange` is written to its node's disk, and the rows
meant for another node go over the network. More rows, or wider rows, mean more bytes written, sent and read.

**Predict:** how many rows go into the `Exchange`? After it, how many tasks get rows?

In [ ]:
from pyspark.sql import Window

w = Window.partitionBy("payment_type")       # one group per payment type: all its rows meet in one task
expensive_window = (trips
                    .withColumn("type_avg", F.avg("total_amount").over(w))
                    .filter(F.col("total_amount") > 2 * F.col("type_avg")))
expensive_window.count()

In [ ]:
sm.capture("expensive trips, window", expensive_window)
sm.stage_table("expensive trips, window")

**How to read it.**

- **Result:** 1,154,571 expensive trips.
- `scan stage`: `rows to Exchange` is 15,096,468, every trip, about 87 `MB to Exchange`.
- `stage after an Exchange`: 3 tasks with rows. `largest task rows` is about 7.8 million: the trips paid by cash. Its
  `longest task s` is the longest in the table.

**Why.** The window needs one output row for every trip, so no task can add up its rows before the `Exchange`
(the **move** check; a group-by can add them up first, as in step 1). And all trips of a payment type go to one task, so the cash trips are a hot key (the **wait**
check). Both problems at once. More nodes don't help: the cash trips stay in one task.

**The fix: group-by, then join.** The window only needs one value per key, each payment type's average.

1. Compute that value with a group-by: 5 rows, one per payment type. Each scan task sends partial totals, as in
   step 1.
2. Join the 5 rows to the trips with a broadcast join (step 2). Each task matches them to its own trips, so no task
   has to process all the trips paid by cash.

In [ ]:
type_avg = (trips.groupBy("payment_type")    # 5 rows: one average per payment type
            .agg(F.avg("total_amount").alias("type_avg")))
expensive_groupby_join = (trips
                          .join(F.broadcast(type_avg), "payment_type")   # each scan task matches the 5 rows to its trips
                          .filter(F.col("total_amount") > 2 * F.col("type_avg")))
expensive_groupby_join.count()

In [ ]:
sm.capture("expensive trips, group-by then join", expensive_groupby_join)
sm.stage_table("expensive trips, group-by then join")

**What changed, and why.**

- The result (output count) is the same: 1,154,571.
- The table has two `scan stage` rows: the file is read twice, once for the averages and once for the join.
- The `stage after an Exchange` row with 40 `rows in` is the group-by's final totals (the 5 averages), not the
  trips: the trips went into no `Exchange`.
- The first `scan stage` sends 40 rows into the `Exchange`, not 15,096,468 (yay!). The second sends only 8 partial counts.
- No task holds the cash trips alone: every task reads about 2 million trips.
- The query is still faster, though it reads the file twice: compare the first lines. **Moving 15.1 million rows, and
  waiting for one task, cost more than a second read.**

A salt would not fix the window here: it still needs every trip of a payment type in one task to put the average on it.

**Step 4. Checks for a slow stage: from what you see to what to change (L6 checks)**

L6's first five checks, and where each one shows in `sm`'s output (L6's **list** and **repeat** need the Spark
UI's Jobs page). On EMR, the Spark UI's stage page shows the same.

| Check | Question | Where it shows here | What to change | Seen in |
|---|---|---|---|---|
| **read** (L5) | how many rows did the stage read to keep the rows the query needs? | checklist `6. read`; `rows in`, scan stage | filter on folder partitions or row-group statistics; read only the columns you need | part A |
| **divide** (L6 P1) | how many tasks read rows, against how many slots? | checklist `reading tasks`; `tasks with rows`, scan stage | more row groups or files | step 1 |
| **wait** (L6 P2) | are the slowest tasks far above the median, or do a few tasks hold most of the rows? | `longest task s` against `median task s`; `largest task rows` against `median task rows` | a salt or a broadcast; for a window, a group-by, then a join | steps 2, 3 |
| **move** (L6 P3) | how much went into the `Exchange`? | `rows to Exchange`, `MB to Exchange` | totals, not rows; broadcast the small table | steps 2, 3 |
| **spill** (L6 P4) | did tasks run out of memory? | `spill to disk MB` (the last note under the table) | more shuffle partitions, or more memory per task | exercise B4 |

**One long task gives you a clue of what will happen with scaling up or repeated queries.**
Check what happens on every stage before diagnosing. Adaptive query execution gets better with spark releases, but it is not perfect, and understanding potential reasons helps.

**Not doing next part right now? Run `%stop_session`! (the last
cell) now. To continue later, run the setup cells and section 1's data cell again and go back to the exercises**

## 2. Exercises

L6's other cases, to work through briefly on your own. 
Taking a pause? Run `%stop_session`, then later run the setup cells and section 1's data cell to start a new session.


For Exercises B1 to B4 Predict, run, read the table (**Look at** names the columns to read).

B5 is a case to discuss. 

Each exercise ends with a commented-out cell holding the answer: open it
after you try.

**Exercise B1. Data stored by key (L6 P2.1)**

If each payment type's trips sit together, a query on one payment type reads only its folder (L4). 

`june-by-type`
is partitioned by payment type, with one file in each and only one row group in each file. 

The query is from step 1.

**Predict:** how many reading tasks? How many rows does the busiest one read?

In [ ]:
sm.tree(problems + "june-by-type/", row_groups=True)

In [ ]:
rides_by_type_folders = (spark.read.parquet(problems + "june-by-type/")
                         .groupBy("payment_type")
                         .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_by_type_folders.collect()

In [ ]:
sm.capture("rides, one file per payment type", rides_by_type_folders)
sm.checklist("rides, one file per payment type")
sm.stage_table("rides, one file per payment type")

**What's up here?** **Look at:** the checklist's files and reading tasks; `tasks with rows`, `largest task rows` and `rows to Exchange` in the `scan stage` row. Which checks are these, and which step had the same cause?

**The fix in L6: write several files per key.** Split each key artificially by adding a salt: give every row a
number below 8, and split by the key and the number. The folders stay one per payment type. We wrote `june-salted`
for you with the code below. Don't run it: it writes to the course bucket.

In [ ]:
# How june-salted was written:
# (trips
#   .withColumn("salt", F.monotonically_increasing_id() % 8)     # or random: F.floor(F.rand() * 8)
#   .repartition("payment_type", "salt")
#   .drop("salt")
#   .write.partitionBy("payment_type")
#   .parquet("s3://.../june-salted/"))

In [ ]:
rides_salted_folders = (spark.read.parquet(problems + "june-salted/")
                        .groupBy("payment_type")
                        .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_salted_folders.collect()

In [ ]:
sm.capture("rides, salted folders", rides_salted_folders)
sm.checklist("rides, salted folders")
sm.stage_table("rides, salted folders")

**What to look at to compare:** the same columns, before and after the salt: the checklist's files and reading tasks; `tasks with rows`, `largest task rows` and `rows to Exchange` in
the `scan stage` row, before and after the salt. What changed?

In [ ]:
# Answer: try the exercise first, then read.
# One file per payment type: 5 folders, 5 files, `4 of 9 scan tasks still hold a row group` (4, not 5: the small
# files of payment types 3 and 4 went into one scan task, as a bundle of small files in part A). `largest task
# rows` 7,799,619: all the cash trips in one task (a file with one row group is read by one task, as in step 1).
# `longest task s` is several times `median task s`. `rows to Exchange` 5. Salted: 5 folders, 26 files, 9 of 9
# reading tasks, `largest task rows` 2,925,130, `rows to Exchange` 18. Checks: divide and wait in the scan
# stage. The plan keeps an `Exchange` in both: Spark doesn't know the folders group the rows. A salt doesn't fix
# a window query (step 3).

**Exercise B2. A salt in a join (L6 P2.2, by hand)**

When both tables are large, broadcast is not a great fix. 
L6's fix by hand: give each trip a random number from 0 to 7, copy every name row 8 times, and join on the key and the number. Each hot key then spreads over up to 8 tasks. 

Cash and card make 2 × 8 = 16 hot (key, number) pairs.

We will set up more shuffle partitions than 16 to make it less likely that several of them land in the same task. 
Glue plans 8 shuffle partitions by default (spark goes with 200), so the cell sets 64. Broadcast is off again, as in step 2, because we want to see what would happen on > 10MiB tables.

**Predict:** after the `Exchange` that gets the trips, how many tasks get rows, and how many rows does the largest task
get?

In [ ]:
shuffle_before = spark.conf.get("spark.sql.shuffle.partitions")    # the session's value, put back below
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "64")

names = (spark.read.schema("payment_type long, payment_name string")    # as in step 2
         .option("header", True).csv("s3://436c-2026w1/l7/layouts/payment_names.csv"))
trips_s = trips.withColumn("salt", F.floor(F.rand() * 8))
names_s = names.crossJoin(spark.range(8).toDF("salt"))
rides_by_name_salted = (trips_s
                        .join(names_s, ["payment_type", "salt"])
                        .groupBy("payment_name")
                        .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_by_name_salted.collect()

In [ ]:
sm.capture("rides by name, salted join", rides_by_name_salted)
sm.stage_table("rides by name, salted join")

**Look at:** the `stage after an Exchange` row with about 15.1 million `rows in`: `tasks with rows` and `largest task
rows`, against step 2's first table. Then the query time against step 2's two runs. Is the salted join faster? Why or
why not?

In [ ]:
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.set("spark.sql.shuffle.partitions", shuffle_before)    # unset would fall back to Spark's 200, not the session's 8

In [ ]:
# Answer: try the exercise first, then read.
# The result is the same as step 2's. The stage that gets the trips has 16 tasks, all with rows (64 were
# planned; adaptive execution merged them into 16), and `largest task rows` about 1 million (the salt is random,
# so the exact count moves a little between runs), against 3 tasks and about 7.8 million without the salt.
# `longest task s` is now shorter (yay again!).

# Every trip still goes into the `Exchange`. In our case, the whole query is not faster: it took about as long
# as step 2 without broadcast in our runs (about 4.5 s). The salt adds work: the 40 copied name rows are built
# and shuffled too, and each trip carries one more column. The hot task took about 2 s, so there was little to
# win in our toy example. The salt pays off when the hot key's task is long; broadcast stays the first fix when
# one table is small.

In [ ]:
# 🔍 Adaptive skew handling (L6 appendix), to try after the exercises. With broadcast off, lower the skew limit from
# 256 MiB to 16 MiB and the target size from 64 MiB to 8 MiB: adaptive execution then splits the cash and card Spark
# partitions by itself.
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
# spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "16m")
# spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "8m")
# rides_by_name_split = (trips
#                        .join(names, "payment_type")
#                        .groupBy("payment_name")
#                        .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
# rides_by_name_split.collect()
#
# then run:
# sm.capture("rides by name, skew split", rides_by_name_split)
# sm.aqe("rides by name, skew split")           # look for "skew join: yes, a hot partition was split"
# sm.stage_table("rides by name, skew split")
#
# and put the settings back:
# spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
# spark.conf.unset("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes")
# spark.conf.unset("spark.sql.adaptive.advisoryPartitionSizeInBytes")

**Exercise B3. A join without broadcast (L6 P3.2)**

The trips joined to the 265-row zone table, then rides and average total per borough. Broadcast is off.

**Predict:** without broadcast, how many tasks get trips after the `Exchange`, and does one of them hold most of the
rows, as in steps 2 and 3?

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

zones = (spark.read.schema("LocationID long, Borough string, Zone string, service_zone string")
         .option("header", True).csv("s3://436c-2026w1/l5/taxi_zone_lookup.csv")
         .select(F.col("LocationID").alias("PULocationID"), "Borough"))
rides_by_borough = (trips
                    .join(zones, "PULocationID")
                    .groupBy("Borough")
                    .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_by_borough.collect()

In [ ]:
sm.capture("rides by borough, broadcast off", rides_by_borough)
sm.stage_table("rides by borough, broadcast off")

In [ ]:
rides_by_borough_broadcast = (trips
                              .join(F.broadcast(zones), "PULocationID")
                              .groupBy("Borough")
                              .agg(F.count("*").alias("rides"), F.avg("total_amount").alias("avg_total")))
rides_by_borough_broadcast.collect()

In [ ]:
sm.capture("rides by borough, broadcast", rides_by_borough_broadcast)
sm.stage_table("rides by borough, broadcast")

In [ ]:
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

**Look at:** `rows to Exchange` and `MB to Exchange` in the `scan stage` row of both tables, and the
`another input` row. Is there a hot key here? Compare `largest task rows` and `median task rows` after the `Exchange`.

In [ ]:
# Answer: try the exercise first, then read.
# Without broadcast: 15,096,468 trips (about 122 MB) and the 265 zones go into an `Exchange`; 8 tasks after it
# do the join in pieces of similar size: `largest task rows` about 2.7 million against a median of about 1.9
# million, so no hot key, just many rows moved (the move check). The join key is the zone, 265 values:
# Manhattan's 13.6 million trips are spread over its zones. With `F.broadcast(zones)`: no stage after an
# `Exchange` for the join; the scan stage sends 64 rows, the group-by's partial totals (8 boroughs from each of
# 8 tasks). Faster with the broadcast: about half the time in our runs (for example 1.6 s against 2.9 s). Same
# fix as step 2, on a table without a hot key.

**Exercise B4. Rows don't fit in memory: spill (L6 P4)**

Each executor sets part of its memory aside for shuffles, joins, sorts and group-bys. When a task's rows don't fit,
it sorts what it holds, writes it to the node's disk and carries on. At the end it reads the spilled rows back and
merges them. The result is correct, but the rows go to disk and back, so the task is slower.

None of our queries spilled: the last note under each stage table says `spill to disk MB: 0 in every stage`. The cell
below forces a spill in step 3's window query. It lowers, on purpose, how many rows the window keeps before spilling to
disk, as if memory were short. These are internal Spark settings.

**Predict:** which stage spills: the scan stage, or the stage after the `Exchange`?

In [ ]:
spark.conf.set("spark.sql.windowExec.buffer.in.memory.threshold", "4096")
spark.conf.set("spark.sql.windowExec.buffer.spill.threshold", "100000")

from pyspark.sql import Window
w = Window.partitionBy("payment_type")
expensive_window_spill = (trips
                          .withColumn("type_avg", F.avg("total_amount").over(w))
                          .filter(F.col("total_amount") > 2 * F.col("type_avg")))
expensive_window_spill.count()

In [ ]:
sm.capture("expensive trips, window, spill forced", expensive_window_spill)
sm.stage_table("expensive trips, window, spill forced")

**Look at:** the last note line, `spill to disk MB`, and the query time against step 3's window. 
Spill only on a stage with a few long tasks points to one cause, spill on tasks of even size to another: which is it here?

In [ ]:
spark.conf.unset("spark.sql.windowExec.buffer.in.memory.threshold")
spark.conf.unset("spark.sql.windowExec.buffer.spill.threshold")

In [ ]:
# Answer: try the exercise first, then read.
# `spill to disk MB`: 96.2, all in the `stage after an Exchange`; the scan stage spilled nothing. That stage's
# two long tasks hold the cash and card trips. Same count (1,154,571), slower than step 3's window, by about 0.5
# to 1 s in our runs. Spill on the few long tasks points to a hot key: deal with the hot key first (step 3's
# group-by, then join). Spill on tasks of even size points to too little memory per task: more shuffle
# partitions, or more memory per task.

**Exercise B5. A new case: bike-share rides (L6, made-up numbers)**

A bike-share company stores a year of rides as Parquet. Each ride gets the average ride time of its start station next
to it, and the query keeps the rides more than 3 times that average. On 16 slots, the stage after the `Exchange` has
12 tasks: the median task takes 0.4 s, the longest 1.9 s, and that task processes 30 % of all rides. No task spilled.

**Discuss:** which check shows the problem, and in which columns of the stage table? Which step ran a query of the
same shape? What would you change, and what would you expect in the stage table after the change?

In [ ]:
# Answer: try the exercise first, then read.
# The wait check: `longest task s` far above `median task s`, and `largest task rows` far above `median task
# rows` (one busy station is a hot key). Same shape as step 3, a window by a key. Change: compute each station's
# average with a group-by, then join it to the rides with a broadcast. After it: `rows to Exchange` drops to
# partial totals, and no task holds 30 % of the rides. Not more row groups (that is the scan stage), not more
# shuffle partitions (no spill, and one station stays in one task), not a salt (the window still needs every
# ride of a station in one task).

In [ ]:
%stop_session